In [ ]:
# Pendapatan, jumlah pesanan, dan pelanggan per bulan
q1 = """
SELECT invoice_month AS bulan,
       ROUND(SUM(total_price), 0) AS pendapatan,
       COUNT(DISTINCT invoice) AS jumlah_pesanan,
       COUNT(DISTINCT customer_id) AS pelanggan_aktif
FROM sales
GROUP BY invoice_month
ORDER BY invoice_month;
"""
display(pd.read_sql(q1, conn))

In [ ]:
# Pertumbuhan bulan ke bulan (window function LAG)
q2 = """
WITH bulanan AS (
  SELECT invoice_month AS bulan, SUM(total_price) AS pendapatan
  FROM sales GROUP BY invoice_month
)
SELECT bulan,
       ROUND(pendapatan, 0) AS pendapatan,
       ROUND((pendapatan - LAG(pendapatan) OVER (ORDER BY bulan)) * 100.0
             / LAG(pendapatan) OVER (ORDER BY bulan), 1) AS growth_pct
FROM bulanan
ORDER BY bulan;
"""
display(pd.read_sql(q2, conn))

In [ ]:
# Top 10 produk berdasarkan pendapatan
q3 = """
SELECT stockcode, MAX(description) AS description,
       SUM(quantity) AS unit_terjual,
       ROUND(SUM(total_price), 0) AS pendapatan
FROM sales
GROUP BY stockcode
ORDER BY pendapatan DESC
LIMIT 10;
"""
display(pd.read_sql(q3, conn))

In [ ]:
# Pendapatan per negara (dengan persentase)
q4 = """
SELECT country,
       ROUND(SUM(total_price), 0) AS pendapatan,
       COUNT(DISTINCT customer_id) AS pelanggan,
       ROUND(SUM(total_price) * 100.0 / (SELECT SUM(total_price) FROM sales), 1) AS persen
FROM sales
GROUP BY country
ORDER BY pendapatan DESC
LIMIT 10;
"""
display(pd.read_sql(q4, conn))

In [ ]:
# 10 pelanggan terbaik dengan peringkat dan kontribusi
q5 = """
SELECT customer_id,
       ROUND(SUM(total_price), 0) AS total_belanja,
       COUNT(DISTINCT invoice) AS jumlah_pesanan,
       RANK() OVER (ORDER BY SUM(total_price) DESC) AS peringkat,
       ROUND(SUM(total_price) * 100.0 / (SELECT SUM(total_price) FROM sales), 2) AS persen_total
FROM sales
GROUP BY customer_id
ORDER BY peringkat
LIMIT 10;
"""
display(pd.read_sql(q5, conn))